# JN2 — The address key

**Curriculum notebook 2 of 6.** A "place" appears differently across sources — `2650 Telegraph Ave`
in one, `2650 Telegraph` in another. To join them you need a **relation**, not string-equality. This
notebook demonstrates the *real* address key the pipeline uses (`s0_keys`), imported — not reinvented.

> Clonable + **read-only**.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


## Config

In [2]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## Import the REAL shared module (don't reinvent the key)

`s0_keys.normalize_address` is the *same* function the whole pipeline uses. A core curriculum rule:
**demonstrate the real tool**, never a teaching toy that drifts from production.

In [3]:
from s0_keys import normalize_address

for a in ['2352 SHATTUCK Ave', '2650 Telegraph Ave', '2650 Telegraph']:
    k = normalize_address(a)
    print(f'{a!r:22} -> {k!r:22}  number={k.number}  street={k.street!r}  stype={k.stype!r}  bucket={k.bucket}')

'2352 SHATTUCK Ave'    -> <2352 SHATTUCK AVE>     number=2352  street='SHATTUCK'  stype='AVE'  bucket=('2352', 'SHATTUCK')
'2650 Telegraph Ave'   -> <2650 TELEGRAPH AVE>    number=2650  street='TELEGRAPH'  stype='AVE'  bucket=('2650', 'TELEGRAPH')
'2650 Telegraph'       -> <2650 TELEGRAPH>        number=2650  street='TELEGRAPH'  stype=None  bucket=('2650', 'TELEGRAPH')


## Trap-lesson: the same place, written differently — you need a RELATION, not `==`

Look at the two Telegraph rows above: `<2650 TELEGRAPH AVE>` and `<2650 TELEGRAPH>`. As **strings**
they are not equal, so a naive `==` join drops the match — and the building silently looks like it
exists in one source but not the other. (This exact bug, in the real comparison, falsely flagged
**357 buildings** as missing until the relation was used.)

The fix is not to *store* one canonical string (you can't make one string equal both "Ave" and bare).
The fix is a **relation**: `AddressKey.matches()` — same number + street, and type-compatible
(equal types, **or either type absent** = wildcard). Two *different present* types do **not** match.

In [4]:
a = normalize_address('2650 Telegraph Ave')
b = normalize_address('2650 Telegraph')        # suffix absent
c = normalize_address('2650 Telegraph Way')    # a DIFFERENT present type

print('Ave  vs (no type) :', a.matches(b), ' <- wildcard: absent type matches present')
print('Ave  vs Way       :', a.matches(c), ' <- two different present types never match')
print('string == (naive) :', repr(a) == repr(b), ' <- the bug a relation avoids')

Ave  vs (no type) : True  <- wildcard: absent type matches present
Ave  vs Way       : False  <- two different present types never match
string == (naive) : False  <- the bug a relation avoids


## The bucket index + the ambiguity guard

`matches()` is a pairwise relation; for speed you index candidates by `bucket = (number, street)`
(type-agnostic) and apply `matches()` within the bucket. But a wildcard has a danger: if a bucket
holds **two** present types (`Ave` *and* `Way`), an absent-type key matches **both** — that is
**ambiguous**, and the rule is *do not auto-match* (flag it). This is the guard that keeps the
wildcard safe.

In [5]:
absent = normalize_address('2650 Telegraph')                 # no street type
candidates = [normalize_address('2650 Telegraph Ave'),
              normalize_address('2650 Telegraph Way')]         # bucket holds TWO present types
matched_types = {c.stype for c in candidates if absent.matches(c)}
ambiguous = absent.stype is None and len(matched_types) > 1
print('absent-type matches present types:', matched_types)
print('ambiguity guard fires (do NOT auto-match):', ambiguous)

absent-type matches present types: {'AVE', 'WAY'}
ambiguity guard fires (do NOT auto-match): True


## Checkpoint

In [6]:
# 1) suffix-present <-> suffix-absent matches
assert normalize_address('2650 Telegraph Ave').matches(normalize_address('2650 Telegraph'))
# 2) two different present types do NOT match
assert not normalize_address('2650 Telegraph Ave').matches(normalize_address('2650 Telegraph Way'))
# 3) the ambiguity guard fires on absent -> multiple present
ab = normalize_address('2650 Telegraph')
pm = {c.stype for c in [normalize_address('2650 Telegraph Ave'),
                        normalize_address('2650 Telegraph Way')] if ab.matches(c)}
assert ab.stype is None and len(pm) > 1

print('CHECKPOINT PASS')
print('  suffix pair matches (wildcard) - two-different-present do NOT - ambiguity guard fires')

CHECKPOINT PASS
  suffix pair matches (wildcard) - two-different-present do NOT - ambiguity guard fires


**JN2 done.** You can now relate the same place across sources safely. **Next — JN3:** the spine and
`net_units`, and the trap where one column means different things on different permit types.